<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2012%20-%202026-05-08%20-%20M%C3%A9tricas%20-%20ENTREGA%20SEMANA%203/clase_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 12 · Métricas y comparación de modelos

**Seminario EDA 2026 — Semana 3, Día 5 (viernes 8 de mayo) · ENTREGA SEMANA 3**

Construiste un modelo. Pide accuracy 85%. ¿Es bueno? Depende. Si lo que predices está balanceado 50/50, es bueno. Si 95% es la clase mayoritaria, entonces 85% es peor que no hacer nada.

Hoy: cómo elegir la métrica correcta. Cómo comparar modelos sin engañarte. Cómo saber si estás overfitting o underfitting.

Este es el día donde entregás el trabajo de la semana. Así que vamos a construir un análisis robusto desde el inicio hasta la conclusión.

> **Hoy haces** · Comparar al menos dos modelos con métricas apropiadas (accuracy, precision, recall, F1, MAE, RMSE) y cross-validation. Es la S3. ~2h.
>
> **Entrega** · **ENTREGA S3 — EDA y Modelado Inicial.** Exploración + hipótesis validadas + primer modelo comparado con baseline, antes del domingo 10 de mayo 23:59 (presentación viernes 8 de mayo en clase).

## Setup

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

print("Setup completo ✓")

---

## El problema real

Tenés 3 modelos para predecir supervivencia en el Titanic. ¿Cuál elegís? Un banco querría precisión (no falsos positivos en fraude). Un hospital querría recall (no perder pacientes en riesgo). La métrica depende del costo de equivocarse.

In [ ]:
titanic = sns.load_dataset("titanic").dropna(subset=["age"]).copy()
titanic["sex_int"] = (titanic["sex"] == "female").astype(int)

X = titanic[["age", "pclass", "sex_int", "sibsp", "parch"]]
y = titanic["survived"]

print(f"Dataset: {X.shape[0]} pasajeros")
print(f"Positivos (supervivientes): {y.sum()} ({100*y.mean():.1f}%)")
print(f"Negativos (fallecidos): {(1-y).sum()} ({100*(1-y).mean():.1f}%)")

---

## 1. Matriz de confusión

La matriz cruza "lo que predije" con "lo real". 4 casos: TP (acertaste positivo), TN (acertaste negativo), FP (dijiste sí pero no), FN (dijiste no pero sí).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)

cm = confusion_matrix(y_te, pred)
print("Matriz de confusión:")
print(cm)
print(f"\nTN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

disp = ConfusionMatrixDisplay(cm, display_labels=["Murió", "Sobrevivió"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión — Titanic")
plt.tight_layout()

### Ejercicio 1

Calcula manualmente accuracy, precision, recall y F1 del modelo anterior.

In [ ]:
tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
total = tn + fp + fn + tp

accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1       : {f1:.3f}")

print(f"\nVerificación con sklearn:")
print(f"Accuracy : {accuracy_score(y_te, pred):.3f}")
print(f"Precision: {precision_score(y_te, pred):.3f}")
print(f"Recall   : {recall_score(y_te, pred):.3f}")
print(f"F1       : {f1_score(y_te, pred):.3f}")

In [ ]:
assert abs(accuracy - accuracy_score(y_te, pred)) < 0.01
print("✅ Ejercicio 1 completado")

---

## 2. Elige la métrica correcta

Escenario: Sos rescatista. Tenés una lista de "probables supervivientes". ¿Qué métrica importa?

- Precision: de los que predije supervivientes, ¿cuántos lo son verdad? (evita falsos positivos)
- Recall: de los verdaderos supervivientes, ¿cuántos detecté? (evita falsos negativos)

En un rescate, recall importa más. No querés dejar gente atrás.

---

## 3. ROC-AUC · ranking vs clases

ROC-AUC mide cuán bien el modelo rankea ejemplos. Un AUC de 0.9 significa: si pesco un positivo y un negativo al azar, hay 90% chance de que el modelo le asigne mayor probabilidad al positivo.

In [ ]:
proba = clf.predict_proba(X_te)[:, 1]
auc_score = roc_auc_score(y_te, proba)

print(f"ROC-AUC = {auc_score:.3f}")

fpr, tpr, _ = roc_curve(y_te, proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], lw=1, linestyle="--", label="Aleatorio (AUC=0.5)")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos (Recall)")
plt.title("Curva ROC")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

---

## 4. Validación cruzada · comparación robusta

Train/test split una vez puede ser suerte. Validación cruzada K-fold entrena k veces, cada vez con un fold distinto como test. Promedia las k métricas. Más confiable.

In [ ]:
clf_lr = LogisticRegression(max_iter=1000, random_state=SEED)

scores_cv = cross_val_score(clf_lr, X, y, cv=5, scoring="roc_auc")

print(f"Scores por fold: {scores_cv}")
print(f"Media: {scores_cv.mean():.3f}")
print(f"Std: {scores_cv.std():.3f}")
print(f"Intervalo: [{scores_cv.mean() - 2*scores_cv.std():.3f}, {scores_cv.mean() + 2*scores_cv.std():.3f}]")

### Ejercicio 2

Compara 3 clasificadores con CV-5: LogisticRegression, DecisionTree (max_depth=5), RandomForest (100 árboles). Métrica: `roc_auc`.

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree (d=5)": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, random_state=SEED),
}

resultados = {}

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc")
    resultados[nombre] = {"mean": scores.mean(), "std": scores.std()}
    print(f"{nombre:30s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}")

df_res = pd.DataFrame(resultados).T
print(f"\nMejor modelo (por AUC media): {df_res['mean'].idxmax()}")

In [ ]:
assert len(resultados) == 3
print("✅ Ejercicio 2 completado")

---

## 5. Visualizar la comparación

In [ ]:
modelos_lista = list(resultados.keys())
means = [resultados[m]["mean"] for m in modelos_lista]
stds = [resultados[m]["std"] for m in modelos_lista]

plt.figure(figsize=(10, 4))
plt.errorbar(modelos_lista, means, yerr=stds, fmt="o-", capsize=5, markersize=8, capthick=2)
plt.ylabel("ROC-AUC")
plt.title("Comparación de modelos con CV-5")
plt.ylim([0.7, 1.0])
plt.grid(alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()

---

## Desafío · ENTREGA SEMANA 3

Construye un modelo de clasificación robusto, compáralo contra alternativas, y elige el mejor justificadamente.

**Checklist:**

1. Preparación de datos (limpieza, features, codificación).
2. Baseline con Logistic Regression (accuracy, precision, recall, F1, ROC-AUC, matriz de confusión).
3. Comparación con 2-3 modelos alternativos usando CV-5.
4. Tabla resumen: modelo, media AUC, std AUC.
5. Elección justificada (métrica principal, estabilidad, trade-offs).
6. Conclusión en prosa (qué aprendiste, cuál es tu modelo final, por qué).

Esto es tu entrega final de la semana 3. Debe ser exhaustivo pero limpio.

In [ ]:
print("\n" + "="*70)
print("ENTREGA SEMANA 3: MODELO Y COMPARACIÓN")
print("="*70)

# === SECCIÓN 1: PREPARACIÓN DE DATOS ===
print("\n1. PREPARACIÓN")
print("-" * 70)
# ← Tu código aquí

# === SECCIÓN 2: BASELINE ===
print("\n2. BASELINE (Logistic Regression)")
print("-" * 70)
# ← Tu código aquí

# === SECCIÓN 3: COMPARACIÓN CON CV-5 ===
print("\n3. COMPARACIÓN DE MODELOS (CV-5)")
print("-" * 70)
# ← Tu código aquí

# === SECCIÓN 4: ELECCIÓN Y JUSTIFICACIÓN ===
print("\n4. MODELO FINAL Y JUSTIFICACIÓN")
print("-" * 70)
# ← Tu código aquí

# === SECCIÓN 5: CONCLUSIÓN ===
print("\n5. CONCLUSIÓN EN PROSA")
print("-" * 70)
# ← Escribe aquí en 3-4 oraciones

print("\n" + "="*70)
print("ENTREGA COMPLETADA")
print("="*70)

---

## Síntesis de Semana 3

**Día 08** → Visualizaciones avanzadas. Matplotlib, Seaborn, Plotly. Las historias que cuentan los datos.

**Día 09** → Regresiones (lineal, polinomial, VIF). El primer modelo predictivo y la multicolinealidad.

**Día 10** → Regresión logística + SMOTE. Clasificación binaria y balanceo de clases.

**Día 11** → Árbol de decisión. Modelo no lineal, Gini/entropía, poda.

**Día 12 (hoy)** → Métricas. Matriz de confusión, AUC, F1, threshold tuning, validación cruzada — la elección justificada del modelo.

Semana 4 viene con dashboards, APIs, presentaciones. Ahora tenés los fundamentos.

> **Recordatorio** · **ENTREGA S3 hoy.** Antes del domingo 10 de mayo 23:59 (presentación oral viernes 8 de mayo) en el repo del equipo.

---

## Trabajo en equipo · ENTREGA FINAL SEMANA 3

Entregan:

1. **Notebook `s3_modelo_final.ipynb`** con:
   - EDA completo (gráficos avanzados)
   - Hipótesis formuladas y testeadas
   - Matriz de correlación + features clave
   - Comparación de 2-3 modelos con CV-5
   - Selección justificada del modelo final

2. **Informe en prosa** (~1-2 páginas):
   - Descripción del dataset
   - Hallazgos principales
   - Hipótesis testadas
   - Métricas del modelo final
   - Próximos pasos

3. **Visualizaciones clave** (mínimo 5):
   - 2-3 gráficos exploratorios
   - 1 gráfico de tests estadísticos
   - 1 matriz de correlación
   - 1 matriz de confusión + ROC

**Plazo:** Lunes 11 de mayo, 23:59 UTC-5

**Evaluación:**
- Código limpio, sin errores, reproducible
- EDA exhaustivo
- Tests correctamente ejecutados
- Modelos entrenados con CV
- Conclusiones en prosa